# 🐱 AnimalMind — ViT Cat Breed Classifier Training Pipeline

Fine-tunes **ViT-Base** on 12 Oxford-IIIT cat breeds with advanced regularizations:
- **Augmentations**: RandAugment, MixUp, CutMix.
- **Regularization**: Label Smoothing (0.1), EMA, Stochastic Depth.
- **Calibration**: Temperature Scaling ($T$) to optimize ECE.
- **HF Hub Export**: Pushes model directly to `firstoff/animalmind-cat-classifier`.

In [1]:
# 1. Verify GPU Acceleration
!nvidia-smi

Wed Jul 29 00:19:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Clone Repository & Install Dependencies
import os
if not os.path.exists('/content/AnimalMind'):
    !git clone https://github.com/firstoff23/AnimalMind.git
%cd /content/AnimalMind/ml_backend
!pip install -q -r requirements_training.txt

Cloning into 'AnimalMind'...
remote: Enumerating objects: 47494, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 47494 (delta 206), reused 236 (delta 143), pack-reused 47171 (from 2)
Receiving objects: 100% (47494/47494), 100.79 MiB | 15.84 MiB/s, done.
Resolving deltas: 100% (17927/17927), done.
/content/AnimalMind/ml_backend


In [3]:
# 3. Set Hugging Face Token (Optional)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("🔑 HF_TOKEN loaded.")
except Exception:
    print("ℹ️ No HF_TOKEN secret configured.")

ℹ️ No HF_TOKEN secret configured.


In [4]:
# 4. Run Cat Breed ViT Fine-Tuning
!python -m training.train_cat_breeds \
    --epochs 30 \
    --batch-size 32 \
    --lr 3e-4 \
    --output-dir models/animalmind-cat-classifier \
    --push-to-hub firstoff/animalmind-cat-classifier

[CatTraining] Starting Cat Breed Fine-Tuning (google/vit-base-patch16-224)...
Cat Breeds: 12 classes -> ['Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British Shorthair', 'Egyptian Mau', 'Maine Coon', 'Persian', 'Ragdoll', 'Russian Blue', 'Siamese', 'Sphynx']
Compute Device: cuda
preprocessor_config.json: 100% 160/160 [00:00<00:00, 1.01MB/s]
config.json: 100% 69.7k/69.7k [00:00<00:00, 93.6MB/s]
[transformers] You passed `num_labels=12` which is incompatible to the `id2label` map of length `1000`.

model.safetensors: downloading bytes:  73% 253M/346M [00:01<00:00, 251MB/s, 21.6MB/s  ]
model.safetensors: downloading bytes:  82% 283M/346M [00:02<00:00, 175MB/s, 25.4MB/s  ]
model.safetensors: downloading bytes:  92% 318M/346M [00:02<00:00, 150MB/s, 26.9MB/s  ]
model.safetensors: downloading bytes: 100% 328M/328M [00:02<00:00, 131MB/s, 29.1MB/s  ]
model.safetensors: reconstructing file: 100% 346M/346M [00:02<00:00, 138MB/s, 31.8MB/s  ]
Loading weights: 100% 200/200 [00:00<00:00, 4982.75it/s]

In [5]:
# 5. Display Calibration Metrics
import json
with open("training/cat_training_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)
print(f"🎯 Val Accuracy : {metrics.get('val_accuracy', 0)*100:.2f}%")
print(f"🌡️ Calibrated T  : {metrics.get('calibrated_temperature', 1.0):.4f}")
print(f"📊 ECE           : {metrics.get('ece', 0.0):.4f}")

🎯 Val Accuracy : 9.17%
🌡️ Calibrated T  : 1.7168
📊 ECE           : 0.0449
